# Tutorial 12: Language Processing

This tutorial explores how ACT-R models language processing, from recognizing individual words to understanding complete narratives. We'll cover word recognition, syntactic parsing, sentence comprehension, language production, reading with eye movements, and dialogue.

## 1. Word Recognition and Frequency Effects

Language processing in ACT-R involves multiple levels: phonology, morphology, syntax, and semantics. At the most basic level, we need to recognize words. Word frequency plays a major role—common words are recognized faster than rare ones.

In [ ]:
import pyactr as actr
import time

word_model = actr.ACTRModel()
word_model.model_parameters['subsymbolic'] = True
word_model.model_parameters['activation_trace'] = True

actr.chunktype("word", "form meaning frequency category")
actr.chunktype("recognition_goal", "state input word")

words = [
    ("cat", "feline_animal", "high", "noun"),
    ("dog", "canine_animal", "high", "noun"),
    ("run", "move_fast", "high", "verb"),
    ("aardvark", "african_mammal", "low", "noun"),
    ("cogitate", "think_deeply", "low", "verb")
]

for form, meaning, freq, cat in words:
    chunk = actr.makechunk(
        chunktype="word",
        form=form,
        meaning=meaning,
        frequency=freq,
        category=cat
    )
    word_model.decmem.add(chunk)

word_model.productionstring(
    name="recognize_word",
    string='''
    =g>
        isa recognition_goal
        state reading
        input =text
    ==>
    =g>
        state retrieving
    +retrieval>
        isa word
        form =text''')

word_model.productionstring(
    name="word_found",
    string='''
    =g>
        isa recognition_goal
        state retrieving
    =retrieval>
        isa word
        form =word
        meaning =meaning
        frequency =freq
    ==>
    =g>
        state recognized
        word =word
        Recognized: =word (=meaning) - =freq frequency''')

print("Word Recognition Model")
print("High frequency words (cat, dog) are recognized faster than low frequency words (aardvark, cogitate)")
print("This is handled automatically in subsymbolic mode through base-level activation")

## 2. Syntactic Parsing

Humans parse sentences incrementally—building up structure word by word rather than waiting for the complete sentence. This model uses a shift-reduce parser with grammar rules to build phrase structure.

In [ ]:
import pyactr as actr

parser_model = actr.ACTRModel()
parser_model.model_parameters['subsymbolic'] = True

actr.chunktype("phrase", "type head modifier complete")
actr.chunktype("parse_state", "position word_list current_phrase stack")
actr.chunktype("grammar_rule", "lhs rhs1 rhs2")

grammar = [
    ("S", "NP", "VP"),
    ("NP", "DET", "N"),
    ("NP", "ADJ", "N"),
    ("VP", "V", "NP"),
    ("VP", "V", "PP"),
    ("PP", "P", "NP")
]

for lhs, rhs1, rhs2 in grammar:
    parser_model.decmem.add(
        actr.makechunk(
            chunktype="grammar_rule",
            lhs=lhs,
            rhs1=rhs1,
            rhs2=rhs2
        )
    )

parser_model.productionstring(
    name="begin_parse",
    string='''
    =g>
        isa parse_state
        position start
        word_list =words
    ==>
    =g>
        position parsing
        current_phrase nil
        stack empty
''')

parser_model.productionstring(
    name="shift_word",
    string='''
    =g>
        isa parse_state
        position parsing
        word_list =first =rest
    ==>
    =g>
        word_list =rest
    +retrieval>
        isa word
        form =first
''')

parser_model.productionstring(
    name="reduce_phrase",
    string='''
    =g>
        isa parse_state
        position parsing
        stack =item1 =item2 =rest
    =retrieval>
        isa grammar_rule
        rhs1 =cat1
        rhs2 =cat2
        lhs =result
    =item1>
        isa phrase
        type =cat1
    =item2>
        isa phrase  
        type =cat2
    ==>
    =g>
        stack =newphrase =rest
    +newphrase>
        isa phrase
        type =result
        head =item2
        modifier =item1
''')

parser_model.productionstring(
    name="resolve_attachment",
    string='''
    =g>
        isa parse_state
        position ambiguous
        current_phrase =pp
    ?retrieval>
        recently_retrieved nil
    ==>
    =g>
        position resolving
    +retrieval>
        isa phrase
        type VP
''')

print("Syntactic Parser")
print("Example: 'The cat chased the mouse in the garden'")
print("The prepositional phrase 'in the garden' can attach to either 'mouse' or 'chased'")
print("Parser must resolve this structural ambiguity")

## 3. Sentence Comprehension

Understanding a sentence requires integrating syntactic structure with semantic content. This model builds propositions from sentences and tracks thematic roles (who did what to whom). It can also handle garden path sentences that require reanalysis.

In [ ]:
import pyactr as actr

comprehension_model = actr.ACTRModel()
comprehension_model.model_parameters['subsymbolic'] = True
comprehension_model.model_parameters['spreading_activation'] = True

actr.chunktype("proposition", "predicate agent patient location time")
actr.chunktype("sentence_goal", "state words meaning syntax")
actr.chunktype("semantic_role", "word role expectation")

comprehension_model.decmem.add(
    actr.makechunk(
        chunktype="proposition",
        predicate="chase",
        agent="cat",
        patient="mouse"
    )
)

comprehension_model.decmem.add(
    actr.makechunk(
        chunktype="semantic_role",
        word="chase",
        role="agent",
        expectation="animate"
    )
)

comprehension_model.productionstring(
    name="integrate_meaning",
    string='''
    =g>
        isa sentence_goal
        state parsing
        words =subj =verb =obj =rest
    =verb_chunk>
        isa word
        form =verb
        category verb
    ==>
    =g>
        state integrating
    +retrieval>
        isa semantic_role
        word =verb
        role agent
''')

comprehension_model.productionstring(
    name="reanalyze_garden_path",
    string='''
    =g>
        isa sentence_goal
        state integrating
        syntax =current_parse
    =retrieval>
        isa semantic_role
        expectation =expected
    ?retrieval>
        state error
    ==>
    =g>
        state reanalyzing
''')

comprehension_model.productionstring(
    name="comprehension_complete",
    string='''
    =g>
        isa sentence_goal
        state integrating
    =meaning>
        isa proposition
        predicate =pred
        agent =agent
        patient =patient
    ==>
    =g>
        state understood
        meaning =meaning
''')

print("Sentence Comprehension Model")
print("\nGarden path example: 'The horse raced past the barn fell'")
print("Initial misparse: 'raced' interpreted as main verb")
print("Reanalysis: 'raced' actually part of reduced relative clause")
print("Correct parse: 'The horse [that was] raced past the barn fell'")

## 4. Language Production

Comprehension is only half the story. Production models how speakers go from concepts to utterances. The process involves multiple stages: conceptual planning, grammatical encoding, lexical selection, and phonological encoding.

In [ ]:
import pyactr as actr

production_model = actr.ACTRModel()
production_model.model_parameters['subsymbolic'] = True

actr.chunktype("message", "intention content urgency")
actr.chunktype("utterance_plan", "stage content syntax words")
actr.chunktype("lexical_item", "concept word frequency accessibility")

concepts = [
    ("dog_concept", "dog", "high", "1.0"),
    ("dog_concept", "hound", "low", "0.5"),
    ("dog_concept", "canine", "low", "0.3"),
    ("big_concept", "big", "high", "1.0"),
    ("big_concept", "large", "medium", "0.7"),
    ("big_concept", "enormous", "low", "0.4")
]

for concept, word, freq, access in concepts:
    production_model.decmem.add(
        actr.makechunk(
            chunktype="lexical_item",
            concept=concept,
            word=word,
            frequency=freq,
            accessibility=access
        )
    )

production_model.productionstring(
    name="formulate_message",
    string='''
    =g>
        isa utterance_plan
        stage conceptualizing
        content =message
    =message>
        isa message
        intention =intent
    ==>
    =g>
        stage grammatical_encoding
''')

production_model.productionstring(
    name="select_word",
    string='''
    =g>
        isa utterance_plan
        stage lexical_access
        content =concept
    ==>
    =g>
        stage retrieving_word
    +retrieval>
        isa lexical_item
        concept =concept
    ''',
    utility=5
)

production_model.productionstring(
    name="prefer_frequent_word",
    string='''
    =g>
        isa utterance_plan
        stage retrieving_word
    =retrieval>
        isa lexical_item
        word =word
        frequency high
    ==>
    =g>
        stage phonological_encoding
        words =word
    ''',
    utility=7
)

production_model.productionstring(
    name="tip_of_tongue",
    string='''
    =g>
        isa utterance_plan
        stage retrieving_word
    ?retrieval>
        state error
    ==>
    =g>
        stage partial_access
''')

production_model.productionstring(
    name="monitor_output",
    string='''
    =g>
        isa utterance_plan
        stage articulating
        words =planned
    ==>
    =g>
        stage monitoring
    +retrieval>
        isa message
        content =intended
''')

print("Language Production Model")
print("\nProduction stages:")
print("- Conceptualization: decide what to say")
print("- Grammatical encoding: build syntactic structure")
print("- Lexical selection: choose words (frequency matters)")
print("- Phonological encoding: plan articulation")
print("- Self-monitoring: detect errors")
print("\nCaptures tip-of-tongue states and word frequency effects in production")

## 5. Reading and Eye Movements

When reading, our eyes don't move smoothly across text—they jump from word to word in saccades. Fixation duration depends on word frequency, length, and predictability. Some words get skipped entirely, and sometimes we regress backward.

In [ ]:
import pyactr as actr
import numpy as np

reading_model = actr.ACTRModel()
reading_model.model_parameters['subsymbolic'] = True

actr.chunktype("text_location", "word position length frequency predictability")
actr.chunktype("reading_goal", "state current_word fixation_time")
actr.chunktype("eye_movement", "type target duration")

sentence = [
    ("The", "1", "3", "high", "high"),
    ("quick", "2", "5", "high", "medium"),
    ("brown", "3", "5", "high", "high"),
    ("fox", "4", "3", "high", "medium"),
    ("jumped", "5", "6", "medium", "low"),
    ("over", "6", "4", "high", "high"),
    ("the", "7", "3", "high", "high"),
    ("lazy", "8", "4", "medium", "medium"),
    ("dog", "9", "3", "high", "high")
]

for word, pos, length, freq, pred in sentence:
    reading_model.decmem.add(
        actr.makechunk(
            chunktype="text_location",
            word=word,
            position=pos,
            length=length,
            frequency=freq,
            predictability=pred
        )
    )

reading_model.productionstring(
    name="identify_word",
    string='''
    =g>
        isa reading_goal
        state fixating
        current_word =pos
    ==>
    =g>
        state processing
    +retrieval>
        isa text_location
        position =pos
''')

reading_model.productionstring(
    name="process_frequent_word",
    string='''
    =g>
        isa reading_goal
        state processing
    =retrieval>
        isa text_location
        word =word
        frequency high
        length =len
    ==>
    =g>
        state planning_saccade
        fixation_time 200
''')

reading_model.productionstring(
    name="process_difficult_word",
    string='''
    =g>
        isa reading_goal
        state processing
    =retrieval>
        isa text_location
        word =word
        frequency low
    ==>
    =g>
        state planning_saccade
        fixation_time 350
''')

reading_model.productionstring(
    name="skip_predictable_word",
    string='''
    =g>
        isa reading_goal
        state planning_saccade
        current_word =pos
    +retrieval>
        isa text_location
        position =pos
        predictability high
        length =len
    ==>
    =g>
        state skipping
        current_word =pos
''')

reading_model.productionstring(
    name="regressive_saccade",
    string='''
    =g>
        isa reading_goal
        state comprehension_failure
        current_word =pos
    ==>
    =g>
        state regressing
        current_word =pos
''')

print("Reading Model with Eye Movements")
print("\nTypical fixation durations:")
print("- High frequency words: ~200ms")
print("- Low frequency words: ~350ms")
print("\nWord skipping more likely for:")
print("- Short words (2-3 letters)")
print("- High predictability words")
print("\nRegressions occur when comprehension fails (~10-15% of saccades)")

## 6. Dialogue and Turn-Taking

Conversation requires coordinating with another person. How do we know when to take our turn? We monitor multiple cues: syntax completion, intonation, pauses, and gaze. This model includes turn-taking, common ground management, and repair mechanisms.

In [ ]:
import pyactr as actr

dialogue_model = actr.ACTRModel()
dialogue_model.model_parameters['subsymbolic'] = True

actr.chunktype("dialogue_state", "speaker turn topic common_ground")
actr.chunktype("speech_act", "type content intent")
actr.chunktype("common_ground", "knowledge shared partner")
actr.chunktype("turn_cue", "type strength signal")

cues = [
    ("syntax_complete", "high", "grammatical_end"),
    ("falling_intonation", "high", "prosodic"),
    ("pause", "medium", "temporal"),
    ("gaze", "medium", "visual"),
    ("gesture_complete", "low", "gestural")
]

for cue_type, strength, signal in cues:
    dialogue_model.decmem.add(
        actr.makechunk(
            chunktype="turn_cue",
            type=cue_type,
            strength=strength,
            signal=signal
        )
    )

dialogue_model.productionstring(
    name="project_turn_end",
    string='''
    =g>
        isa dialogue_state
        speaker other
        turn listening
    ==>
    =g>
        turn monitoring
    +retrieval>
        isa turn_cue
        Monitoring for turn-taking cues...
''')

dialogue_model.productionstring(
    name="take_turn",
    string='''
    =g>
        isa dialogue_state
        turn monitoring
    =retrieval>
        isa turn_cue
        strength high
        signal =signal
    ==>
    =g>
        speaker self
        turn planning
''')

dialogue_model.productionstring(
    name="update_common_ground",
    string='''
    =g>
        isa dialogue_state
        common_ground =cg
    =speech>
        isa speech_act
        type assertion
        content =info
    ==>
    =g>
        common_ground =updated_cg
    +updated_cg>
        isa common_ground
        knowledge =info
        shared yes
''')

dialogue_model.productionstring(
    name="request_clarification",
    string='''
    =g>
        isa dialogue_state
        turn planning
    ?retrieval>
        state error
    ==>
    =g>
        turn speaking
    +speech>
        isa speech_act
        type question
        content clarification_needed
        intent repair
''')

dialogue_model.productionstring(
    name="produce_backchannel",
    string='''
    =g>
        isa dialogue_state
        speaker other
        turn listening
    =speech>
        isa speech_act
        type continuing
    ==>
    =g>
        turn backchanneling
''')

print("Dialogue Model")
print("\nTurn-taking typically happens with ~200ms gaps between speakers")
print("Relies on projecting turn end from multiple cues")
print("\nCommon ground: tracking what's shared knowledge vs. new information")
print("Repair: clarification requests when understanding fails")
print("Backchannels: 'uh-huh', 'mm-hmm' to signal continued attention")

## 7. Story Comprehension

Understanding narratives requires integrating everything we've covered: word recognition, parsing, semantic integration, and inference. This model builds a situation model of the story and generates bridging inferences to maintain coherence.

In [ ]:
import pyactr as actr

story_model = actr.ACTRModel()
story_model.model_parameters['subsymbolic'] = True
story_model.model_parameters['spreading_activation'] = True

actr.chunktype("story_element", "type content position causal_link")
actr.chunktype("situation_model", "protagonist goal obstacle outcome time location")
actr.chunktype("inference", "type premise conclusion certainty")
actr.chunktype("story_goal", "state current_sentence situation_model inferences")

story_model.decmem.add(
    actr.makechunk(
        chunktype="story_element",
        type="setting",
        content="introduces_character_location"
    )
)

story_model.decmem.add(
    actr.makechunk(
        chunktype="story_element",
        type="complication",
        content="introduces_problem"
    )
)

story_model.productionstring(
    name="build_situation_model",
    string='''
    =g>
        isa story_goal
        state reading
        current_sentence =sent
        situation_model =model
    =sent>
        isa sentence
        content =content
    ==>
    =g>
        state updating_model
    +retrieval>
        isa story_element
        content =content
''')

story_model.productionstring(
    name="make_causal_inference",
    string='''
    =g>
        isa story_goal
        state updating_model
        situation_model =model
    =model>
        isa situation_model
        goal =goal
        obstacle =obstacle
    ==>
    =g>
        state inferring
    +inference>
        isa inference
        type causal
        premise goal_blocked
        conclusion character_motivated
        certainty high
''')

story_model.productionstring(
    name="bridge_information_gap",
    string='''
    =g>
        isa story_goal
        state inferring
        current_sentence =sent
    =sent>
        isa sentence
        referent ambiguous
    ==>
    =g>
        state bridging
    +retrieval>
        isa situation_model
        protagonist =char
''')

story_model.productionstring(
    name="predict_outcome",
    string='''
    =g>
        isa story_goal
        state inferring
    =model>
        isa situation_model
        goal =goal
        obstacle overcome
    ==>
    =g>
        state predicting
    +inference>
        isa inference
        type predictive
        conclusion goal_achieved
        certainty medium
''')

story_model.productionstring(
    name="detect_incoherence",
    string='''
    =g>
        isa story_goal
        state updating_model
    ?retrieval>
        state error
    ==>
    =g>
        state reprocessing
''')

print("Story Comprehension Model")
print("\nExample:")
print("'Sarah wanted to buy a bicycle. The store was closed.")
print("She waited outside in the rain. Finally, the owner arrived.")
print("She bought the red bicycle she had been dreaming about.'")
print("\nThe model:")
print("- Builds situation model (protagonist: Sarah, goal: buy bicycle)")
print("- Tracks causal links (closed store blocks goal)")
print("- Makes bridging inferences ('She' refers to Sarah)")
print("- Generates predictions based on story structure")
print("- Detects and repairs incoherence")

## Exercises

1. Build a model that resolves lexical ambiguity (e.g., "bank" as financial institution vs. river edge) using context.

2. Model metaphor comprehension—how do people understand "Time is money" through conceptual mapping?

3. Create a second language learning model with L1 interference patterns.

4. Model text summarization by extracting and prioritizing main ideas.

5. Add realistic speech disfluencies (um, uh, false starts, repairs) to the production model.

## Summary

Language processing in ACT-R involves multiple interacting components:

- Words are retrieved from declarative memory with frequency effects
- Syntactic parsing happens incrementally, word by word
- Semantic integration builds propositional representations
- Production goes from concepts through grammatical encoding to articulation
- Eye movements during reading reflect ongoing comprehension processes
- Dialogue requires coordination through turn-taking and common ground

The models in this tutorial demonstrate how ACT-R's architecture—with its declarative memory, productions, and subsymbolic parameters—can capture the complexity of human language use.